# Naviguer dans le bucket de la Romanche

L'accès est dans `bucket.py` (config dans `.env`). Arborescence : `station/année/mois/jour/image.jpg`.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import matplotlib.pyplot as plt
from bucket import Bucket

b = Bucket()
b

Bucket('romanche-mathis', prefix='', endpoint=AWS)

## 1. Lister un dossier
`b.ls(prefix)` renvoie les sous-dossiers et les images du niveau demandé. Prefix vide = racine.

In [ ]:
def show(prefix=''):
    folders, files = b.ls(prefix)
    print(f'/{prefix}')
    for f in folders: print('  [dir]', f)
    for o in files[:20]: print('  [img]', o['Key'], o['Size'] // 1024, 'Ko')
    if len(files) > 20: print(f'  ... {len(files)} images au total')
    return folders, files

show()

In [ ]:
show('romanche-01/')

In [ ]:
show('romanche-01/2026/11/')

## 2. Descendre automatiquement jusqu'au dernier jour
À chaque niveau on prend le dernier dossier (ordre alphabétique = ordre chronologique).

In [ ]:
prefix = ''
while True:
    folders, files = b.ls(prefix)
    if not folders:
        break
    prefix = folders[-1]
print('Dernier dossier :', prefix, '-', len(files), 'images')

## 3. Afficher des images
Charge en mémoire (sans écrire sur disque).

In [ ]:
sel = files[-6:]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, o in zip(axes.flat, sel):
    ax.imshow(b.load_image(o['Key']))
    ax.set_title(o['Key'].rsplit('/', 1)[-1], fontsize=9)
    ax.axis('off')
plt.tight_layout()

## 4. Aller à un jour précis
Change `jour` et relance.

In [ ]:
jour = 'romanche-01/2026/11/03/'
folders, files = b.ls(jour)
print(len(files), 'images')
if files:
    plt.figure(figsize=(8, 6))
    plt.imshow(b.load_image(files[len(files)//2]['Key']))
    plt.title(files[len(files)//2]['Key'])
    plt.axis('off')

## 5. Télécharger
Vers `data/raw/`, en gardant l'arborescence. Saute ce qui est déjà là.

In [ ]:
paths = b.download(files)
print(len(paths), 'fichiers dans data/raw/')